## Mobility Robustness Optimization (MRO)

Takes in new observation data to train or update the bayesian digital twin models. It processes the input data and updates the model to better reflect the current network conditions.

Then MRO optimizes the mobility robustness by solving the underlying problem using the trained model: finding optimal `HYST` and `TTT`. There are two solve approaches shown: 

- Simple MRO
- Reinforced MRO

In [1]:
import sys
from pathlib import Path
sys.path.append(f"{Path().absolute().parent}")

In [2]:
import pandas as pd
import numpy as np
from apps.mobility_robustness_optimization.simple_mro import SimpleMRO
from apps.mobility_robustness_optimization.mro_rl import ReinforcedMRO
from apps.mobility_robustness_optimization.mro_ml import BayesianMRO

*unzip the `data/mro_data.zip` file to get `data/mro_data/` folder*

# Showcasing **Simple MRO** Solving Approach

use the following example `topology` and `mobility_model_params` to initiate MRO.

In [3]:
topology = pd.read_csv('data/mro_data/mro_topology.csv')
topology

,cell_lat,cell_lon,cell_id,cell_az_deg,cell_carrier_freq_mhz
0,-90.0,-180.0,cell_1,0,2100
1,0.0,0.0,cell_2,120,2100
2,90.0,180.0,cell_3,240,2100


In [4]:
mobility_model_params = {
    "ue_tracks_generation": {
            "params": {
                "simulation_duration": 3600,
                "simulation_time_interval_seconds": 0.01,
                "num_ticks": 50,
                "num_batches": 1,
                "ue_class_distribution": {
                    "stationary": {
                        "count": 10,
                        "velocity": 0,
                        "velocity_variance": 1
                    },
                    "pedestrian": {
                        "count": 5,
                        "velocity": 2,
                        "velocity_variance": 1
                    },
                    "cyclist": {
                        "count": 5,
                        "velocity": 5,
                        "velocity_variance": 1
                    },
                    "car": {
                        "count": 12,
                        "velocity": 20,
                        "velocity_variance": 1
                    }
                },
                "lat_lon_boundaries": {
                    "min_lat": -90,
                    "max_lat": 90,
                    "min_lon": -180,
                    "max_lon": 180
                },
                "gauss_markov_params": {
                    "alpha": 0.5,
                    "variance": 0.8,
                    "rng_seed": 42,
                    "lon_x_dims": 100,
                    "lon_y_dims": 100,
                    "// TODO": "Account for supporting the user choosing the anchor_loc and cov_around_anchor.",
                    "// Current implementation": "the UE Tracks generator will not be using these values.",
                    "// anchor_loc": {},
                    "// cov_around_anchor": {}
            }
        }
    }
}

<b>Optionally,</b> use mobility model to get `alpha` of your data and set it to params.

In [5]:
# [OPTIONAL] run this cell to get alpha calculated from the data into mobility_model_params

from radp.digital_twin.mobility.param_regression import get_predicted_alpha

# 20 UEs x 50 ticks = 1000 rows
ue_data = pd.read_csv("data/mro_data/UE_data_20UE_100ticks.csv") # change this to the data you want to use
ue_data = ue_data.rename(columns={'latitude': 'lat', 'longitude': 'lon'})

# set random initial alpha
alpha0 = np.random.choice(np.arange(0, 1.1, 0.1))

alpha = get_predicted_alpha(ue_data, alpha0 = alpha0, seed = 42)

print(f"Learned alpha: {alpha:.2f}\n")

mobility_model_params["ue_tracks_generation"]["params"]["gauss_markov_params"]["alpha"] = alpha

Learned alpha: 0.50



/Users/tanzimfarhan/Desktop/Maveric/maveric/radp/digital_twin/mobility/param_regression.py:145: FutureWarning: Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)
  df = df.groupby("mock_ue_id").apply(calculate_distances_and_velocities)


In [ ]:
mro = SimpleMRO(mobility_model_params, topology)

In [15]:
# initially bayesian_digital_twins is empty
print(f"bayesian_digital_twins: {mro.bayesian_digital_twins}")

bayesian_digital_twins: {}


- prepare `new_data` for training/updating `bayesian_digital_twins`

    - `new_data` should have received power data in cartesian df format. required cols ['latitude', 'longitude', 'cell_id', 'cell_rxpwr_dbm']

In [6]:
# 20 UEs x 100 ticks x 3 cells cartesian = 6000 rows
ue_data_with_rxpower = pd.read_csv("data/mro_data/UE_data_with_rxpower_20UE_100ticks_train.csv") # change this to the data you want to use
input_data = ue_data_with_rxpower.copy()

input_data.head()

,longitude,latitude,cell_id,cell_rxpwr_dbm
0,-22.625309,59.806764,1,-100.311970
1,-22.625309,59.806764,2,-99.841523
2,-22.625309,59.806764,3,-99.432278
3,119.764151,54.857584,1,-100.294405
4,119.764151,54.857584,2,-100.132420


In [17]:
# train bayesian_digital_twins from scratch
mro.train_or_update_rf_twins(new_data=input_data)

[2025-07-13 21:53:05,309] INFO:  Iter 1/100 - Loss: 0.784 (delta=inf)


No Bayesian Digital Twins available for update. Training from scratch.


[2025-07-13 21:53:05,344] INFO:  Iter 2/100 - Loss: 0.765 (delta=-0.019081)
[2025-07-13 21:53:05,383] INFO:  Iter 3/100 - Loss: 0.745 (delta=-0.019100)
[2025-07-13 21:53:05,420] INFO:  Iter 4/100 - Loss: 0.726 (delta=-0.019148)
[2025-07-13 21:53:05,456] INFO:  Iter 5/100 - Loss: 0.707 (delta=-0.019222)
[2025-07-13 21:53:05,491] INFO:  Iter 6/100 - Loss: 0.688 (delta=-0.019364)
[2025-07-13 21:53:05,523] INFO:  Iter 7/100 - Loss: 0.668 (delta=-0.019518)
[2025-07-13 21:53:05,555] INFO:  Iter 8/100 - Loss: 0.648 (delta=-0.019725)
[2025-07-13 21:53:05,592] INFO:  Iter 9/100 - Loss: 0.629 (delta=-0.019932)
[2025-07-13 21:53:05,633] INFO:  Iter 10/100 - Loss: 0.608 (delta=-0.020143)
[2025-07-13 21:53:05,664] INFO:  Iter 11/100 - Loss: 0.588 (delta=-0.020348)
[2025-07-13 21:53:05,698] INFO:  Iter 12/100 - Loss: 0.567 (delta=-0.020547)
[2025-07-13 21:53:05,728] INFO:  Iter 13/100 - Loss: 0.547 (delta=-0.020700)
[2025-07-13 21:53:05,761] INFO:  Iter 14/100 - Loss: 0.526 (delta=-0.020930)
[2025-0


Bayesian Digital Twins trained successfully.


In [ ]:
mro.bayesian_digital_twins # bayesian_digital_twins is trained for each cell_id

can save trained/updated `bayesian_digital_twins`

In [ ]:
saving_dir_relative_path = "data/mro_data/"

mro.save_bdt(saving_dir_relative_path) # True indicates save is successful

call `solve()` method to get optimized `HYST` and `TTT`

In [ ]:
# adjust n_epochs for better performance
hyst, ttt = mro.solve(n_epochs=100)

In [ ]:
from notebooks.radp_library import mro_plot_scatter, plot_sinr_db_by_ue, mro_score_3d_plot
from radp.digital_twin.utils.constants import RLF_THRESHOLD
from radp.digital_twin.utils.cell_selection import perform_attachment_hyst_ttt

attached_df = perform_attachment_hyst_ttt(mro.simulation_data, hyst, ttt, RLF_THRESHOLD)
mro_plot_scatter(attached_df, topology)

In [ ]:
mro_score_3d_plot(mro.score)

In [ ]:
ue_id = 0 # change this to the UE you want to plot
plot_sinr_db_by_ue(attached_df, mro.simulation_data, ue_id)

can load this `bayesian_digital_twins` later when needed

In [ ]:
mro.bayesian_digital_twins = {} # bayesian_digital_twins is empty again

pkl_file_path = "data/mro_data/digital_twins.pkl"
mro.load_bdt(pkl_file_path) # True indicates load is successful

In [ ]:
# Dummy solve call to avoid fantasy observation error: ensuring all test independent caches exist
mro.solve(n_epochs=2)

let's try updating the `bayesian_digital_twins` with new observations

In [ ]:
# 20 UEs x 100 ticks x 3 cells cartesian = 6000 rows
new_obeservations = pd.read_csv("data/mro_data/UE_data_with_rxpower_20UE_100ticks_update.csv") # change this to the data you want to use
input_data = new_obeservations.copy()

input_data.head()

In [ ]:
# update bdt with new observations, calling train_or_update_rf_twin() again
mro.train_or_update_rf_twins(input_data)

can solve with updated `bayesian_digital_twins`

In [ ]:
# adjust n_epochs for better performance
mro.solve(n_epochs=100)

# Showcasing **Reinforced MRO** Solving Approach

use the following example `topology` and `mobility_model_params` to initiate MRO.

In [ ]:
rl_mro = ReinforcedMRO(mobility_model_params, topology)

In [ ]:
print(f"bayesian_digital_twins: {rl_mro.bayesian_digital_twins}", end='\n\n') # bayesian_digital_twins is empty initially

pkl_file_path = "data/mro_data/digital_twins.pkl" # run previous section to have this file
rl_mro.load_bdt(pkl_file_path) # True indicates load is successful

In [ ]:
rl_mro.bayesian_digital_twins

- load `new_data` for training/updating `bayesian_digital_twins`

    - `new_data` should have received power data in cartesian df format. required cols ['latitude', 'longitude', 'cell_id', 'cell_rxpwr_dbm']

In [ ]:
# 20 UEs x 50 ticks x 3 cells cartesian = 3000 rows
ue_data_with_rxpower = pd.read_csv("data/mro_data/UE_data_with_rxpower_20UE_50ticks.csv")
input_data = ue_data_with_rxpower.copy()

input_data

In [ ]:
# Dummy solve call to avoid fantasy observation error: ensuring all test independent caches exist
rl_mro.solve(total_timesteps=2)

In [ ]:
# update bdt with new data
rl_mro.train_or_update_rf_twins(input_data)

Solve using updated `bayesian_digital_twins`

In [ ]:
# adjust total_timesteps for better performance
hyst, ttt = rl_mro.solve(total_timesteps=100)

In [ ]:
attached_df = perform_attachment_hyst_ttt(mro.simulation_data, hyst, ttt, RLF_THRESHOLD)
mro_plot_scatter(attached_df, topology)

In [ ]:
ue_id = 0 # change this to the UE you want to plot
plot_sinr_db_by_ue(attached_df, mro.simulation_data, ue_id)

# Showcasing **Bayesian or XGBoost MRO** Solving Approach

use the following example `topology` and `mobility_model_params` to initiate MRO.

In [7]:
mro = BayesianMRO(mobility_model_params, topology)


In [8]:
mro.train_or_update_rf_twins(input_data)

[2025-07-13 22:03:43,922] INFO:  Iter 1/100 - Loss: 0.784 (delta=inf)
[2025-07-13 22:03:43,959] INFO:  Iter 2/100 - Loss: 0.765 (delta=-0.019080)
[2025-07-13 22:03:43,993] INFO:  Iter 3/100 - Loss: 0.745 (delta=-0.019101)


No Bayesian Digital Twins available for update. Training from scratch.


[2025-07-13 22:03:44,036] INFO:  Iter 4/100 - Loss: 0.726 (delta=-0.019151)
[2025-07-13 22:03:44,095] INFO:  Iter 5/100 - Loss: 0.707 (delta=-0.019225)
[2025-07-13 22:03:44,164] INFO:  Iter 6/100 - Loss: 0.688 (delta=-0.019359)
[2025-07-13 22:03:44,227] INFO:  Iter 7/100 - Loss: 0.668 (delta=-0.019512)
[2025-07-13 22:03:44,268] INFO:  Iter 8/100 - Loss: 0.648 (delta=-0.019742)
[2025-07-13 22:03:44,303] INFO:  Iter 9/100 - Loss: 0.629 (delta=-0.019925)
[2025-07-13 22:03:44,334] INFO:  Iter 10/100 - Loss: 0.608 (delta=-0.020137)
[2025-07-13 22:03:44,368] INFO:  Iter 11/100 - Loss: 0.588 (delta=-0.020349)
[2025-07-13 22:03:44,401] INFO:  Iter 12/100 - Loss: 0.567 (delta=-0.020548)
[2025-07-13 22:03:44,433] INFO:  Iter 13/100 - Loss: 0.547 (delta=-0.020711)
[2025-07-13 22:03:44,466] INFO:  Iter 14/100 - Loss: 0.526 (delta=-0.020913)
[2025-07-13 22:03:44,499] INFO:  Iter 15/100 - Loss: 0.505 (delta=-0.021102)
[2025-07-13 22:03:44,534] INFO:  Iter 16/100 - Loss: 0.483 (delta=-0.021266)
[2025


Bayesian Digital Twins trained successfully.


In [9]:
mro.solve()


Optimized Hyst: 2.568939405978036,
Optimized TTT: 4


(2.568939405978036, 4)